# ADS 505 Technical Notebook: Predicting U.S. Flight Departure Delays

**Team:** 4 (individual project, instructor-approved)  
**Author:** Michael Rosalia  
**GitHub repository:** https://github.com/mrosalia96/ads505-flight-delay-prediction

Predict pre-departure delay risk (≥ 15 minutes) for flights originating at SAN, LAX, PHX, DEN, and SEA in 2023, using BTS on-time performance and NOAA weather. Sections are added as the analysis progresses.

In [ ]:
# All package imports live in this first code cell.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BTS_DIR = RAW_DIR / "bts"
NOAA_DIR = RAW_DIR / "noaa"

ORIGIN_AIRPORTS = ("SAN", "LAX", "PHX", "DEN", "SEA")
ANALYSIS_YEAR = 2023

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from build_flights_sample import build_sample  # noqa: E402

print("Project root:", PROJECT_ROOT)

## 1. Problem statement and data description

**Problem.** Airlines and airports often learn about departure delays as they unfold. This project predicts **pre-departure delay risk** (15+ minutes late) so operations leaders can prioritize staffing, buffers, and passenger communication before pushback.

**Data.** U.S. DOT BTS Reporting Carrier On-Time Performance for calendar year 2023, filtered to origin airports SAN, LAX, PHX, DEN, and SEA. NOAA weather will be joined in a later step. Acquisition details and APA citations: `docs/data_sources.md`.

**How data are obtained (reproducibility).** This is **not** a live filtered API pull. Peers download the same public monthly ZIP files from fixed TranStats URLs (`src/download_bts.py`), then run `src/build_flights_sample.py` to keep only our five origins. Raw ZIPs and the processed parquet stay on disk (gitignored); the repo stores the scripts and docs so anyone can rebuild the sample and re-run this notebook.

**Target.** Prefer BTS `DepDel15` (1 = departure delay ≥ 15 minutes). Rows with missing delay outcome or cancelled flights are handled explicitly below.

In [ ]:
# Build (or rebuild) the filtered origin sample from monthly BTS ZIPs.
# Requires: python src/download_bts.py --year 2023  (raw ZIPs in data/raw/bts/)
sample_path = build_sample(year=ANALYSIS_YEAR, origins=ORIGIN_AIRPORTS)
print("Sample file:", sample_path)

In [ ]:
# Load the processed sample and report basic shape / delay rate.
flights = pd.read_parquet(sample_path) if sample_path.suffix == ".parquet" else pd.read_csv(
    sample_path, parse_dates=["FlightDate"]
)

# Modelling set: exclude cancelled flights; require a defined 15-min delay flag.
mask_usable = (flights["Cancelled"] == 0) & flights["DepDel15"].notna()
flights_model = flights.loc[mask_usable].copy()
flights_model["DepDel15"] = flights_model["DepDel15"].astype(int)

print(f"Raw filtered rows: {len(flights):,}")
print(f"Usable (not cancelled, DepDel15 known): {len(flights_model):,}")
print(f"Date range: {flights_model['FlightDate'].min().date()} → {flights_model['FlightDate'].max().date()}")
print("Rows by origin:")
print(flights_model["Origin"].value_counts().reindex(list(ORIGIN_AIRPORTS)))
print(f"Overall DepDel15 rate: {flights_model['DepDel15'].mean():.3%}")
flights_model.head()

## 2. Exploratory data analysis (EDA)

Charts appear only where discussed. Focus: where delay risk concentrates by **airport**, **month**, and **scheduled departure hour** — patterns an operations lead would recognize without weather yet.

### 2.1 Delay rate by origin airport

**Takeaway:** DEN has the highest 15+ minute departure-delay rate (~27%), while SEA is lowest (~17%). SAN sits near the middle (~20%). A single national average would hide that hub-level risk differs by roughly 10 percentage points across this sample.

In [ ]:
delay_by_origin = (
    flights_model.groupby("Origin", as_index=False)["DepDel15"]
    .mean()
    .rename(columns={"DepDel15": "delay_rate"})
)
delay_by_origin["Origin"] = pd.Categorical(
    delay_by_origin["Origin"], categories=list(ORIGIN_AIRPORTS), ordered=True
)
delay_by_origin = delay_by_origin.sort_values("Origin")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=delay_by_origin, x="Origin", y="delay_rate", color="#4C72B0", ax=ax)
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Origin airport")
ax.set_title("Departure delay rate by origin (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
for bar, rate in zip(ax.patches, delay_by_origin["delay_rate"]):
    ax.annotate(f"{rate:.1%}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
delay_by_origin

### 2.2 Seasonality by month

**Takeaway:** Delay risk peaks in **June–July** (~28–29%) and is lowest in **November** (~15%). Summer operational pressure shows up clearly in this five-airport sample; a model that ignores month will miss a large, predictable swing.

In [ ]:
delay_by_month = (
    flights_model.groupby("Month", as_index=False)["DepDel15"]
    .mean()
    .rename(columns={"DepDel15": "delay_rate"})
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=delay_by_month, x="Month", y="delay_rate", marker="o", ax=ax)
ax.set_xticks(range(1, 13))
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Month")
ax.set_title("Departure delay rate by month (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_ylim(0, max(0.35, delay_by_month["delay_rate"].max() * 1.15))
plt.tight_layout()
plt.show()
delay_by_month

### 2.3 Time-of-day pattern

**Takeaway:** Early banks are comparatively reliable (roughly **7–11%** delayed at 05:00–07:00 scheduled), while late afternoon / evening banks reach about **28–31%**. That asymmetry matters for staffing: the same airport needs different attention at 6 a.m. vs 6 p.m.

In [ ]:
flights_model = flights_model.copy()
flights_model["crs_dep_hour"] = (flights_model["CRSDepTime"] // 100).astype("Int64")

delay_by_hour = (
    flights_model.dropna(subset=["crs_dep_hour"])
    .groupby("crs_dep_hour", as_index=False)
    .agg(delay_rate=("DepDel15", "mean"), n_flights=("DepDel15", "size"))
)
# Keep hours with enough volume for a stable rate
delay_by_hour = delay_by_hour[delay_by_hour["n_flights"] >= 500]

fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=delay_by_hour, x="crs_dep_hour", y="delay_rate", marker="o", ax=ax)
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Scheduled departure hour (local)")
ax.set_title("Departure delay rate by scheduled hour (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()
delay_by_hour.sort_values("delay_rate", ascending=False).head(5)

### 2.4 EDA implications for modeling

Airport, month, and scheduled hour are strong candidate features (and stratification checks). Next steps: join origin weather, then preprocess / split **by time** so summer peaks are not leaked into winter test months incorrectly.